# PHẦN 4: HUẤN LUYỆN XẾP HẠNG VÀ ĐÁNH GIÁ

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost', 'pyarrow', 'polars'])
import os, gc
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import xgboost as xgb
import polars as pl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Cập nhật đường dẫn theo chuẩn Kaggle
INPUT_DIR = '/kaggle/input/datasets/b22dckh072/feature-engineering/' 
TRAIN_PATH = os.path.join(INPUT_DIR, 'train_interactions.parquet')
META_PATH  = os.path.join(INPUT_DIR, 'filtered_metadata.parquet')
CAND_PATH  = os.path.join(INPUT_DIR, 'candidates_phase2.parquet')
FEAT_OUT   = os.path.join(INPUT_DIR, 'features.parquet')
TEST_PATH  = os.path.join(INPUT_DIR, 'test_interactions.parquet')

# KHAI BÁO CỐ ĐỊNH DANH SÁCH ĐẶC TRƯNG 
FEATURES = ['user_orders', 'item_sales', 'price', 'sasrec_rank', 'lightgcn_rank']

In [ ]:
print("Đang chuẩn bị luồng dữ liệu Out-of-core...")
class ParquetIter(xgb.core.DataIter):
    def __init__(self, path, features):
        self.path = path
        self.features = features
        self.pf = pq.ParquetFile(self.path)
        # File 600MB khá nhẹ, tăng batch_size lên 200k để GPU ăn dữ liệu nhanh hơn
        self.it = self.pf.iter_batches(batch_size=200000) 
        super().__init__()
        
    def next(self, input_data):
        try:
            batch = next(self.it)
            chunk = batch.to_pandas()
            input_data(data=chunk[self.features], label=chunk['label'])
            return 1
        except StopIteration:
            return 0
            
    def reset(self):
        self.it = self.pf.iter_batches(batch_size=200000)

it = ParquetIter(FEAT_OUT, FEATURES)
dtrain = xgb.ExtMemQuantileDMatrix(it, missing=np.nan)

print("Bắt đầu huấn luyện với GPU...")
params = {
    'tree_method': 'hist', 
    'device': 'cuda',             # Kích hoạt GPU
    'objective': 'binary:logistic', 
    'eval_metric': 'logloss', 
    'max_depth': 6,               
    'scale_pos_weight': 4,        
    'learning_rate': 0.1
}

# XGBoost chạy GPU cực nhanh, tự tin để 50 vòng (hoặc 100 nếu muốn)
model = xgb.train(params, dtrain, num_boost_round=50) 
print("Huấn luyện hoàn tất!")

# Trực quan hóa độ quan trọng
xgb.plot_importance(model, importance_type='gain')
plt.show()

In [ ]:
model_path = '/kaggle/working/xgboost_ranking_model.json'
model.save_model(model_path)
print(f"Đã lưu mô hình tại: {model_path}")

In [ ]:
if 'model' not in locals():
    print("Đang nạp mô hình từ file...")
    model = xgb.Booster()
    model.load_model('/kaggle/working/xgboost_ranking_model.json')

In [ ]:
print("Đang tạo bảng đặc trưng nền bằng Polars (Lazy Evaluation)...")
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_meta = pl.scan_parquet(META_PATH)

# Thống kê User và Item (Bảng này cực nhẹ)
df_use = lf_train.group_by('mapped_user_id').len().rename({"len": "user_orders"}).collect()
df_ite = lf_train.group_by('mapped_item_id').len().rename({"len": "item_sales"}).collect()

# Trích xuất và dọn dẹp giá (Price)
mapping_df = lf_train.select(['parent_asin', 'mapped_item_id']).unique()
df_price = (
    lf_meta.select(['parent_asin', 'price'])
    .join(mapping_df, on='parent_asin', how='inner')
    .with_columns(
        pl.col('price').str.replace_all(r'[^0-9.]', '').cast(pl.Float32, strict=False).fill_null(0.0)
    )
    .select(['mapped_item_id', 'price'])
    .unique()
    .collect()
)

# Chuyển sang Pandas để gộp từng chunk cho ổn định
df_use_pd = df_use.to_pandas()
df_ite_pd = df_ite.to_pandas()
df_price_pd = df_price.to_pandas()

del df_use, df_ite, df_price, lf_train, lf_meta, mapping_df
gc.collect()

print("Bắt đầu chấm điểm ứng viên từ file 5.5GB...")
pf = pq.ParquetFile(CAND_PATH)
reader = pf.iter_batches(batch_size=500000) 
all_top100 = []

for batch in tqdm(reader, desc="Scoring Chunks"):
    chunk = batch.to_pandas()
    
    # Nối đặc trưng
    chunk = chunk.merge(df_use_pd, on='mapped_user_id', how='left')
    chunk = chunk.merge(df_ite_pd, on='mapped_item_id', how='left')
    chunk = chunk.merge(df_price_pd, on='mapped_item_id', how='left')
    chunk.fillna({'price': 0, 'user_orders': 0, 'item_sales': 0}, inplace=True)
    
    # Chuẩn bị dữ liệu và chấm điểm
    X_cands = chunk[FEATURES]
    dtest = xgb.DMatrix(X_cands, missing=np.nan)
    chunk['score'] = model.predict(dtest)
    
    # Lấy Top 100 nội bộ chunk để giảm RAM ngay lập tức
    sub = chunk[['mapped_user_id', 'mapped_item_id', 'score']]
    sub = sub.sort_values(['mapped_user_id', 'score'], ascending=[True, False])
    all_top100.append(sub.groupby('mapped_user_id').head(100))
    
    del chunk, X_cands, dtest, sub
    gc.collect()

print("Đang tổng hợp Top 100 cuối cùng...")
final_cands = pd.concat(all_top100, ignore_index=True)
# Sắp xếp lại để đảm bảo mỗi user chỉ lấy đúng 100 món tốt nhất toàn cục
final_cands = final_cands.sort_values(['mapped_user_id', 'score'], ascending=[True, False])
top100 = final_cands.groupby('mapped_user_id').head(100)

# --- PHẦN LƯU FILE QUAN TRỌNG ---
FINAL_TOP100_PATH = '/kaggle/working/top100_final_recommendations.parquet'
top100.to_parquet(FINAL_TOP100_PATH, index=False)
print(f"ĐÃ LƯU KẾT QUẢ TOP 100 TẠI: {FINAL_TOP100_PATH}")
# -------------------------------

del all_top100, final_cands
gc.collect()
print('Hoàn tất xếp hạng và lưu trữ an toàn!')

In [ ]:
df_test = pd.read_parquet(TEST_PATH)
truth = df_test.groupby('mapped_user_id')['mapped_item_id'].apply(set)

top100_ranked = top100.copy()
top100_ranked['rank'] = top100_ranked.groupby('mapped_user_id').cumcount()

truth_df = df_test[['mapped_user_id', 'mapped_item_id']].rename(columns={'mapped_item_id': 'true_item'})

hits = pd.merge(top100_ranked, truth_df, on='mapped_user_id')
hits = hits[hits['mapped_item_id'] == hits['true_item']]

valid_users = truth.index
n_valid = len(valid_users)

hits_at10 = hits[hits['rank'] < 10]
HR10 = hits_at10['mapped_user_id'].nunique() / n_valid

hits_at10 = hits_at10.copy()
hits_at10['ndcg'] = 1.0 / np.log2(hits_at10['rank'] + 2)
NDCG10 = hits_at10.groupby('mapped_user_id')['ndcg'].sum().reindex(valid_users, fill_value=0).mean()

truth_sizes = truth.apply(len)
recall_per_user = hits.groupby('mapped_user_id').size() / truth_sizes
Recall100 = recall_per_user.reindex(valid_users, fill_value=0).mean()

print(f"HR@10: {HR10:.4f}")
print(f"NDCG@10: {NDCG10:.4f}")
print(f"Recall@100: {Recall100:.4f}")